# Mock 6C · ARC-AGI-2: Abstract Reasoning · Suy luận trừu tượng trên lưới màu

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | Each task shows a few **demonstration pairs** (input grid → output grid) that all follow one hidden rule. Work out the rule and produce the **output grid** for each test input: its size and every cell's colour. | Mỗi bài cho vài **cặp ví dụ** (lưới vào → lưới ra) cùng tuân theo một quy luật ẩn. Hãy tìm ra quy luật và tạo **lưới kết quả** cho từng lưới test: cả kích thước lẫn màu của từng ô. |
| **Data** | `train/` 1,000 tasks with all answers · `public_test/` 60 tasks (85 test inputs) · `private_test/` 60 tasks (87 test inputs), answers hidden. Grids are at most 30×30, colours are digits 0-9. | `train/` 1.000 bài có đủ đáp án · `public_test/` 60 bài (85 lưới test) · `private_test/` 60 bài (87 lưới test), đáp án được ẩn. Lưới tối đa 30×30, màu là chữ số 0-9. |
| **Metric** | pass@2 accuracy: for every test input you may give **2 attempts**; it counts as solved if one of them is **exactly** right (same size, every cell). Score = share of test inputs solved, **higher is better**. Ranking uses the **private** score. | Độ chính xác pass@2: mỗi lưới test được **2 lần thử**; tính là đúng nếu một trong hai **khớp hoàn toàn** (đúng kích thước, đúng mọi ô). Điểm = tỷ lệ lưới test giải đúng, **càng cao càng tốt**. Xếp hạng theo điểm **private**. |
| **Submission** | `public_submission.csv`, `private_submission.csv` with columns `id,attempt_1,attempt_2`, one row per test input, `id` = `<task_id>_<test_index>`. A grid is written row by row between pipes: `[[0,1],[1,0]]` → `\|01\|10\|`. | `public_submission.csv`, `private_submission.csv` gồm cột `id,attempt_1,attempt_2`, mỗi lưới test một dòng, `id` = `<task_id>_<số thứ tự test>`. Lưới được ghi theo từng hàng giữa các dấu gạch đứng: `[[0,1],[1,0]]` → `\|01\|10\|`. |
| **Rules** | Any method and any library (program search, neural networks, LLMs you run yourself...). Use only the provided files: **no external data**, and **no looking up the public ARC answers** (the test tasks are public ARC-AGI-2 evaluation tasks in disguise: honour code). **No solving test tasks by hand**: every answer must come from your code. | Được dùng mọi phương pháp và thư viện (tìm kiếm chương trình, mạng nơ-ron, LLM tự chạy...). Chỉ dùng các file được cung cấp: **không dữ liệu ngoài** và **không tra đáp án ARC công khai** (các bài test là bài đánh giá ARC-AGI-2 công khai đã được ngụy trang: hãy giữ danh dự). **Không tự giải tay bài test**: mọi đáp án phải do code của bạn sinh ra. |

$$\text{Score}=\frac{1}{N}\sum_{i=1}^{N}\mathbb{1}\big[\text{attempt}_{1,i}=y_i\ \lor\ \text{attempt}_{2,i}=y_i\big]$$

**Files · Các file** (one JSON object each · mỗi file là một đối tượng JSON): `task_id → {"train": [{"input": grid, "output": grid}, ...], "test": [{"input": grid}, ...]}`.
In `train/train.json` the test pairs also have their `output` · Trong `train/train.json` các cặp test có cả `output`.

🇬🇧 ARC is hard: the best Kaggle entry of ARC Prize 2025 solved about a quarter of the hidden tasks, while humans solve most of them. A few percent here is already a good result.
🇻🇳 ARC rất khó: đội mạnh nhất trên Kaggle ở ARC Prize 2025 giải được khoảng một phần tư số bài ẩn, trong khi con người giải được phần lớn. Vài phần trăm ở đây đã là kết quả tốt.

Data · Dữ liệu: [ARC-AGI-2](https://github.com/arcprize/ARC-AGI-2) by the ARC Prize Foundation, Apache-2.0, from the original files of the Kaggle competition [ARC Prize 2026 - ARC-AGI-2](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-2) (see `dataset/LICENSE`, `dataset/README.md`).
Inspired by · Lấy cảm hứng từ: [ARC Prize 2026 - ARC-AGI-2 (Kaggle)](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-2).

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import json
from collections import Counter

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Hyperparameters · Cấu hình và siêu tham số
🇬🇧 Nothing to fill in here: the data downloads itself, and your team comes from your account on the site when you submit (section 7).
🇻🇳 Không cần điền gì ở đây: dữ liệu tự tải về, và đội của bạn được lấy từ tài khoản trên trang web khi nộp bài (phần 7).

In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "6C"
FOLDER  = "6C_arc_agi_2"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/mock-6"

# Already have the folder? Point this at the parent of 6C_arc_agi_2/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 6C_arc_agi_2/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path + ".part",
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        os.replace(zip_path + ".part", zip_path)  # only a complete download gets the final name
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/6C_arc_agi_2/dataset"
TRAIN_JSON = f"{DATA_DIR}/train/train.json"
PUBLIC_JSON = f"{DATA_DIR}/public_test/public_test.json"
PRIVATE_JSON = f"{DATA_DIR}/private_test/private_test.json"
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
N_VALIDATION_TASKS = 1000  # training tasks used to measure the solver / số bài train dùng để đánh giá solver
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading and Visualisation · Đọc và hiển thị dữ liệu

In [ ]:
def load(path):
    with open(path) as f:
        return json.load(f)


train_tasks = load(TRAIN_JSON)
public_tasks = load(PUBLIC_JSON)
private_tasks = load(PRIVATE_JSON)
for name, tasks in [("Train", train_tasks), ("Public test", public_tasks), ("Private test", private_tasks)]:
    print(f"{name:12s}: {len(tasks):4d} tasks, {sum(len(t['test']) for t in tasks.values()):4d} test inputs")

# EN: the 10 standard ARC colours (0 = black) / VI: 10 màu chuẩn của ARC (0 = đen)
ARC_CMAP = ListedColormap(["#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00",
                           "#AAAAAA", "#F012BE", "#FF851B", "#7FDBFF", "#870C25"])


def show_task(task, title=""):
    """EN: draw the demonstration pairs (top: input, bottom: output) and the test inputs.
    VI: vẽ các cặp ví dụ (trên: đầu vào, dưới: đầu ra) và các lưới test."""
    pairs = task["train"] + task["test"]
    fig, axes = plt.subplots(2, len(pairs), figsize=(2.2 * len(pairs), 4.4), squeeze=False)
    for j, p in enumerate(pairs):
        for i, key in enumerate(["input", "output"]):
            ax = axes[i, j]
            ax.set_xticks([]); ax.set_yticks([])
            if key in p:
                ax.imshow(np.array(p[key]), cmap=ARC_CMAP, vmin=0, vmax=9)
            ax.set_title(("demo " if j < len(task["train"]) else "test ") + key, fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


first_id = sorted(train_tasks)[0]
show_task(train_tasks[first_id], f"train task {first_id}")
first_public = sorted(public_tasks)[0]
show_task(public_tasks[first_public], f"public test task {first_public} (output hidden / đáp án ẩn)")

## 4. Solver: a Tiny Program Search · Bộ giải: tìm kiếm chương trình đơn giản

In [ ]:
def grid_to_str(grid):
    """[[0,1],[1,0]] -> '|01|10|' (submission format / định dạng nộp bài)."""
    return "|" + "|".join("".join(str(int(v)) for v in row) for row in grid) + "|"


GEOMETRIC = {  # EN: the 8 rotations / flips of a grid. VI: 8 phép xoay / lật một lưới.
    "identity": lambda g: g,
    "rot90": lambda g: np.rot90(g, 1),
    "rot180": lambda g: np.rot90(g, 2),
    "rot270": lambda g: np.rot90(g, 3),
    "flip_lr": lambda g: g[:, ::-1],
    "flip_ud": lambda g: g[::-1, :],
    "transpose": lambda g: g.T,
    "anti_transpose": lambda g: np.rot90(g, 2).T,
}


def crop_to_content(g):
    """EN: cut out the bounding box of every cell that is not the background (most common colour).
    VI: cắt khung bao quanh mọi ô khác màu nền (màu phổ biến nhất)."""
    bg = Counter(g.ravel().tolist()).most_common(1)[0][0]
    rows, cols = np.where(g != bg)
    if len(rows) == 0:
        return g
    return g[rows.min():rows.max() + 1, cols.min():cols.max() + 1]


def candidate_programs(task):
    """EN: a tiny library of whole-grid programs (name, function).
    VI: một thư viện nhỏ các chương trình biến đổi cả lưới (tên, hàm)."""
    progs = [(name, f) for name, f in GEOMETRIC.items()]
    progs += [(f"crop+{name}", (lambda f: lambda g: f(crop_to_content(g)))(f)) for name, f in GEOMETRIC.items()]
    # EN: output k times larger than the input in the first demo -> try "zoom" and "tile"
    # VI: đầu ra lớn gấp k lần đầu vào ở ví dụ đầu -> thử "phóng to" và "lát gạch"
    x0, y0 = np.array(task["train"][0]["input"]), np.array(task["train"][0]["output"])
    if y0.shape[0] % x0.shape[0] == 0 and y0.shape[1] % x0.shape[1] == 0:
        a, b = y0.shape[0] // x0.shape[0], y0.shape[1] // x0.shape[1]
        if (a, b) != (1, 1):
            progs.append((f"zoom{a}x{b}", lambda g: np.kron(g, np.ones((a, b), dtype=g.dtype))))
            progs.append((f"tile{a}x{b}", lambda g: np.tile(g, (a, b))))
    return progs


def learn_colour_map(preds, outs):
    """EN: a colour -> colour table that turns every prediction into its output, or None.
    VI: bảng đổi màu biến mọi dự đoán thành đầu ra tương ứng, hoặc None nếu không có."""
    table = {}
    for p, o in zip(preds, outs):
        if p.shape != o.shape:
            return None
        for a, b in zip(p.ravel().tolist(), o.ravel().tolist()):
            if table.setdefault(a, b) != b:
                return None
    return table


def apply_colour_map(g, table):
    lut = np.arange(10)
    for a, b in table.items():
        lut[a] = b
    return lut[g]


def solve_task(task):
    """EN: keep every program (optionally followed by a colour map) that reproduces ALL demonstration pairs,
    then apply the first two to each test input. Returns one (attempt_1, attempt_2) per test input.
    VI: giữ mọi chương trình (có thể kèm bảng đổi màu) tái tạo đúng TẤT CẢ các cặp ví dụ, rồi áp dụng
    hai chương trình đầu tiên cho từng lưới test. Trả về (attempt_1, attempt_2) cho mỗi lưới test."""
    xs = [np.array(p["input"]) for p in task["train"]]
    ys = [np.array(p["output"]) for p in task["train"]]
    tests = [np.array(p["input"]) for p in task["test"]]
    found = []  # list of functions test_grid -> output grid
    for name, f in candidate_programs(task):
        try:
            preds = [f(x) for x in xs]
            if all(p.shape == y.shape and (p == y).all() for p, y in zip(preds, ys)):
                found.append(f)
                continue
            table = learn_colour_map(preds, ys)
            if table is not None:
                found.append((lambda f, t: lambda g: apply_colour_map(f(g), t))(f, table))
        except Exception:  # a program that does not apply to this grid is simply skipped
            continue

    # EN: fallback guesses: the test input itself, then the most common output size filled with the most common colour
    # VI: dự phòng: chính lưới test, rồi kích thước đầu ra phổ biến nhất tô bằng màu phổ biến nhất
    shape = Counter(y.shape for y in ys).most_common(1)[0][0]
    colour = Counter(np.concatenate([y.ravel() for y in ys]).tolist()).most_common(1)[0][0]
    answers = []
    for t in tests:
        guesses = []
        for f in found:
            try:
                out = np.asarray(f(t))
            except Exception:
                continue
            if out.ndim == 2 and 1 <= out.shape[0] <= 30 and 1 <= out.shape[1] <= 30:  # a valid ARC grid
                g = grid_to_str(out)
                if g not in guesses:
                    guesses.append(g)
        for g in (grid_to_str(t), grid_to_str(np.full(shape, colour))):
            if g not in guesses:
                guesses.append(g)
        answers.append((guesses[0], guesses[1]))
    return answers


print("Solver ready / Solver đã sẵn sàng:", len(GEOMETRIC), "geometric programs + crop, zoom, tile, colour maps")

## 5. Validation on the Training Tasks · Đánh giá trên các bài train

In [ ]:
def pass_at_2(tasks, n=None):
    """EN: the competition metric on tasks whose answers we know (the training set).
    VI: độ đo của cuộc thi trên các bài đã biết đáp án (tập train)."""
    hits, total = 0, 0
    for tid in sorted(tasks)[:n]:
        answers = solve_task(tasks[tid])
        for (a1, a2), pair in zip(answers, tasks[tid]["test"]):
            truth = grid_to_str(pair["output"])
            hits += truth in (a1, a2)
            total += 1
    return hits / total, hits, total


t0 = datetime.now()
acc, hits, total = pass_at_2(train_tasks, N_VALIDATION_TASKS)
print(f"Train pass@2: {acc:.4f} ({hits}/{total} test inputs) in {(datetime.now() - t0).total_seconds():.1f}s")
print("EN: the hidden test tasks are much harder than most training tasks: expect a far lower test score.")
print("VI: các bài test ẩn khó hơn nhiều so với phần lớn bài train: điểm test sẽ thấp hơn nhiều.")

## 6. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
def export(tasks, csv_path):
    """Save <id, attempt_1, attempt_2> CSV (one row per test input) and zip it.
    Lưu CSV <id, attempt_1, attempt_2> (mỗi lưới test một dòng) và nén zip."""
    rows = []
    for tid in sorted(tasks):
        for k, (a1, a2) in enumerate(solve_task(tasks[tid])):
            rows.append({"id": f"{tid}_{k}", "attempt_1": a1, "attempt_2": a2})
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} ({len(rows)} rows) and {zip_path}")


export(public_tasks, "public_submission.csv")
export(private_tasks, "private_submission.csv")
pd.read_csv("public_submission.csv").head()

## 7. Submit · Nộp bài

🇬🇧 Your team comes from your account on [seleccion.ftds.online](https://seleccion.ftds.online/), not from a name typed here. The cell below prints a short code once per Colab session; type it on the site and this notebook submits as you.

🇻🇳 Đội của bạn lấy từ tài khoản trên [seleccion.ftds.online](https://seleccion.ftds.online/). Ô bên dưới in ra một mã ngắn mỗi phiên Colab; nhập mã đó trên trang web.

In [ ]:
def check_submission(csv_path, tasks):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path, dtype=str)
    want = {f"{tid}_{k}" for tid, t in tasks.items() for k in range(len(t["test"]))}
    assert list(sub.columns) == ["id", "attempt_1", "attempt_2"], f"{csv_path}: columns must be id,attempt_1,attempt_2"
    assert len(sub) == len(want) and set(sub["id"]) == want, f"{csv_path}: ids do not match the test inputs"
    for col in ["attempt_1", "attempt_2"]:
        ok = sub[col].fillna("").str.fullmatch(r"\|(?:[0-9]{1,30}\|){1,30}")
        assert ok.all(), f"{csv_path}: {col} has invalid grids, e.g. id {sub.loc[~ok, 'id'].iloc[0]} / lưới không hợp lệ"


check_submission("public_submission.csv", public_tasks)
check_submission("private_submission.csv", private_tasks)

In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "6C"          # which problem this notebook is for
CONTEST = "mock-6"    # this notebook is for Mock Contest 6
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
    print(f"   {BASE}/")